# 02 — Fetch Products

Pulls the full OneBill product catalog and every product's price-plan
detail. Pure fetch/catalog step — matching against vBill plan codes happens
separately in `03_Match_Plan_Codes.ipynb`.

**Flow**

1. `GET /rest/ProductService/v1/products` — every product code.
2. `GET /rest/ProductService/v1/products/{code}` for each one.
   - If OneBill returns validation error `10PR1126` ("Product is not
     available for the user."), the product is **not** available to this
     API user — recorded (name + code) but with no usable price plans.
   - Otherwise, every entry in `pricePlanInfos[]` becomes one row of
     `df_available_priceplans`.

**Output**
- `migration_data/02_available_priceplans.csv`
- `migration_data/02_unavailable_products.csv`
- `migration_data/02_failed_product_lookups.csv`

## 1. Setup

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403

logger = get_logger("fetch_products")
session = new_session(max_workers=10)


python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 27
python-dotenv could not parse statement starting at line 32
python-dotenv could not parse statement starting at line 38
python-dotenv could not parse statement starting at line 44


In [2]:
# HARDCODED_TOKEN = "133cc573-dc96-4136-9834-92e1d3c8b55e"

# token_manager._token = HARDCODED_TOKEN
# token_manager._expires_at = datetime.now() + timedelta(hours=1)  # adjust to match the real token's actual TTL

## 2. Fetch the full product list

In [3]:
df_products_all = fetch_all_products(session)
logger.info(f"Fetched {len(df_products_all):,} products from ProductService")
df_products_all.head()


2026-07-29 05:46:55,302 [INFO] Fetched 0 products from ProductService


""


## 3. Fetch each product's detail

Separates products into available / unavailable / failed — see the module
docstring above for what each means.

> **Response shape**: the single-product endpoint returns the product
> object **directly** — `data` *is* the product, unlike the list endpoint
> which wraps everything in `{"product": [...]}`. `pricePlanInfos` sits
> right on `data`.

In [4]:
available_rows: list[dict] = []       # one row per (product, pricePlanInfo)
unavailable_rows: list[dict] = []      # one row per product with no usable plans
failed_rows: list[dict] = []

codes = df_products_all["code"].dropna().unique().tolist()
logger.info(f"Fetching detail for {len(codes):,} product codes...")

for i, code in enumerate(codes, start=1):
    status, data, message = fetch_product_detail(session, code)

    if status == "available":
        product = data or {}
        plans = product.get("pricePlanInfos", [])
        if not plans:
            unavailable_rows.append({
                "product_name": product.get("name"),
                "product_code": product.get("code", code),
                "reason": "available product, but no pricePlanInfos returned",
            })
        else:
            for plan in plans:
                available_rows.append({
                    "product_name":    product.get("name"),
                    "product_code":    product.get("code", code),
                    "priceplan_name":  plan.get("name"),
                    "priceplan_code":  plan.get("code"),
                    "priceplan_id":    plan.get("id"),
                })

    elif status == "unavailable":
        list_row = df_products_all.loc[df_products_all["code"] == code]
        name = list_row["name"].iloc[0] if not list_row.empty else None
        unavailable_rows.append({"product_name": name, "product_code": code, "reason": message})

    else:  # failed
        failed_rows.append({"product_code": code, "error": message})
        logger.error(f"  [FAIL] product {code} — {message}")

    if i % 50 == 0 or i == len(codes):
        logger.info(f"Progress: {i}/{len(codes)} product codes checked")

logger.info(
    f"Done — {len(available_rows):,} available price-plan rows, "
    f"{len(unavailable_rows):,} unavailable products, {len(failed_rows):,} failed lookups"
)


2026-07-28 10:14:53,046 [INFO] Fetching detail for 3 product codes...
2026-07-28 10:14:56,056 [INFO] Progress: 3/3 product codes checked
2026-07-28 10:14:56,057 [INFO] Done — 1 available price-plan rows, 2 unavailable products, 0 failed lookups


## 4. Build dataframes

In [5]:
df_available_priceplans = pd.DataFrame(
    available_rows, columns=["product_name", "product_code", "priceplan_name", "priceplan_code", "priceplan_id"]
)
df_unavailable_products = pd.DataFrame(unavailable_rows, columns=["product_name", "product_code", "reason"])
df_failed_products = pd.DataFrame(failed_rows, columns=["product_code", "error"])

logger.info(
    f"{len(df_available_priceplans):,} available price-plan rows, "
    f"{len(df_unavailable_products):,} unavailable products, "
    f"{len(df_failed_products):,} failed lookups"
)
df_available_priceplans.head(20)


2026-07-28 10:15:04,379 [INFO] 1 available price-plan rows, 2 unavailable products, 0 failed lookups


,product_name,product_code,priceplan_name,priceplan_code,priceplan_id
0,Test Product,PROD302,Test plan2,code124,302


## 5. Save

In [ ]:
save_df("products_available", df_available_priceplans)
save_df("products_unavailable", df_unavailable_products)
save_df("products_failed", df_failed_products)


Saved 1 rows -> migration_data\02_available_priceplans.csv
Saved 2 rows -> migration_data\02_unavailable_products.csv
Saved 0 rows -> migration_data\02_failed_product_lookups.csv
